In [ ]:
!pip install datasets transformers manga109api Pillow torch torchvision ultralytics

from google.colab import userdata
from datasets import load_dataset
import zipfile
import manga109api
import os

from ultralytics import YOLO
from PIL import Image

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))

In [ ]:
from huggingface_hub import snapshot_download

path = snapshot_download(
    repo_id="hal-utokyo/Manga109",
    repo_type="dataset",
    local_dir="/content/manga109"
)
print(path)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

/content/manga109


In [ ]:
for root, dirs, files in os.walk("/content/manga109"):
    level = root.replace("/content/manga109", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:  # only show files for top 2 levels
        for f in files[:5]:  # limit to 5 files per folder
            print(f"{indent}  {f}")

manga109/
  Manga109_released_2023_12_07.zip
  README.md
  .gitattributes
  .cache/
    huggingface/
      download/


In [ ]:
zip_path = "/content/manga109/Manga109_released_2023_12_07.zip"

with zipfile.ZipFile(zip_path, 'r') as z:
    print(z.namelist()[:20])  # preview contents first

['Manga109_released_2023_12_07/', '__MACOSX/._Manga109_released_2023_12_07', 'Manga109_released_2023_12_07/books.txt', '__MACOSX/Manga109_released_2023_12_07/._books.txt', 'Manga109_released_2023_12_07/annotations.v2020.12.18/', '__MACOSX/Manga109_released_2023_12_07/._annotations.v2020.12.18', 'Manga109_released_2023_12_07/images/', '__MACOSX/Manga109_released_2023_12_07/._images', 'Manga109_released_2023_12_07/annotations.v2018.05.31/', '__MACOSX/Manga109_released_2023_12_07/._annotations.v2018.05.31', 'Manga109_released_2023_12_07/annotations/', '__MACOSX/Manga109_released_2023_12_07/._annotations', 'Manga109_released_2023_12_07/readme.txt', '__MACOSX/Manga109_released_2023_12_07/._readme.txt', 'Manga109_released_2023_12_07/annotations_COO/', '__MACOSX/Manga109_released_2023_12_07/._annotations_COO', 'Manga109_released_2023_12_07/annotations_Manga109Dialog/', '__MACOSX/Manga109_released_2023_12_07/._annotations_Manga109Dialog', 'Manga109_released_2023_12_07/annotations.v2020.12.18/M

In [ ]:
extract_path = "/content/manga109_data"

with zipfile.ZipFile(zip_path, 'r') as z:
    members = [m for m in z.namelist() if not m.startswith("__MACOSX")]
    z.extractall(extract_path, members=members)

print("Done!")

Done!


In [ ]:
data_root = "/content/manga109_data/Manga109_released_2023_12_07"
api = manga109api.Parser(root_dir=data_root)

# List all books
print(api.books)

['ARMS', 'AisazuNihaIrarenai', 'AkkeraKanjinchou', 'Akuhamu', 'AosugiruHaru', 'AppareKappore', 'Arisa', 'BEMADER_P', 'BakuretsuKungFuGirl', 'Belmondo', 'BokuHaSitatakaKun', 'BurariTessenTorimonocho', 'ByebyeC-BOY', 'Count3DeKimeteAgeru', 'DollGun', 'Donburakokko', 'DualJustice', 'EienNoWith', 'EvaLady', 'EverydayOsakanaChan', 'GOOD_KISS_Ver2', 'GakuenNoise', 'GarakutayaManta', 'GinNoChimera', 'Hamlet', 'HanzaiKousyouninMinegishiEitarou', 'HaruichibanNoFukukoro', 'HarukaRefrain', 'HealingPlanet', 'HeiseiJimen', 'HighschoolKimengumi_vol01', 'HighschoolKimengumi_vol20', 'HinagikuKenzan', 'HisokaReturns', 'JangiriPonpon', 'JijiBabaFight', 'Joouari', 'Jyovolley', 'KarappoHighschool', 'KimiHaBokuNoTaiyouDa', 'KoukouNoHitotachi', 'KuroidoGanka', 'KyokugenCyclone', 'LancelotFullThrottle', 'LoveHina_vol01', 'LoveHina_vol14', 'MAD_STONE', 'MadouTaiga', 'MagicStarGakuin', 'MagicianLoad', 'MariaSamaNihaNaisyo', 'MayaNoAkaiKutsu', 'MemorySeijin', 'MeteoSanStrikeDesu', 'MiraiSan', 'MisutenaideDaisy'

In [ ]:
import os
from PIL import Image

LABEL2ID = {"body": 0, "text": 1}

def convert_all_annotations(api, data_root, label_out_root):
    image_dir = os.path.join(data_root, "images")

    for book in api.books:
        ann = api.get_annotation(book=book)

        for page in ann['page']:
            page_index = page['@index']
            W = page['@width']
            H = page['@height']

            img_filename = f"{page_index:03d}.jpg"
            img_path = os.path.join(image_dir, book, img_filename)
            if not os.path.exists(img_path):
                continue

            lines = []
            for tag, label_id in LABEL2ID.items():
                for item in page[tag]:
                    x1 = item['@xmin']
                    y1 = item['@ymin']
                    x2 = item['@xmax']
                    y2 = item['@ymax']

                    cx = ((x1 + x2) / 2) / W
                    cy = ((y1 + y2) / 2) / H
                    w  = (x2 - x1) / W
                    h  = (y2 - y1) / H
                    lines.append(f"{label_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

            label_out = os.path.join(label_out_root, book)
            os.makedirs(label_out, exist_ok=True)
            with open(os.path.join(label_out, img_filename.replace(".jpg", ".txt")), "w") as f:
                f.write("\n".join(lines))

    print("Annotation conversion done!")

label_out_root = "/content/manga109_labels"
convert_all_annotations(api, data_root, label_out_root)

Annotation conversion done!


In [ ]:
# Verify a label file looks right
import random
book = random.choice(api.books)
label_file = f"/content/manga109_labels/{book}/002.txt"
if os.path.exists(label_file):
    with open(label_file) as f:
        print(f.read())

0 0.238210 0.289744 0.430472 0.418803
0 0.119710 0.783333 0.163241 0.429915
0 0.357920 0.121368 0.101572 0.241026
1 0.095828 0.823504 0.085248 0.147863
1 0.262394 0.811966 0.067715 0.126496
1 0.385127 0.553846 0.067715 0.097436


In [ ]:
import shutil, yaml, os

data_root = "/content/manga109_data/Manga109_released_2023_12_07"
image_dir = os.path.join(data_root, "images")
label_out_root = "/content/manga109_labels"
dataset_root = "/content/manga109_dataset"

# Train/val split (80/20)
all_books = api.books
split = int(len(all_books) * 0.8)
train_books = all_books[:split]
val_books = all_books[split:]

print(f"Train: {len(train_books)} books, Val: {len(val_books)} books")

# Copy images and labels into dataset folder
for split_name, books in [("train", train_books), ("val", val_books)]:
    img_split_dir = os.path.join(dataset_root, "images", split_name)
    lbl_split_dir = os.path.join(dataset_root, "labels", split_name)
    os.makedirs(img_split_dir, exist_ok=True)
    os.makedirs(lbl_split_dir, exist_ok=True)

    for book in books:
        src_img = os.path.join(image_dir, book)
        src_lbl = os.path.join(label_out_root, book)
        dst_img = os.path.join(img_split_dir, book)
        dst_lbl = os.path.join(lbl_split_dir, book)

        if os.path.exists(src_img) and not os.path.exists(dst_img):
            shutil.copytree(src_img, dst_img)
        if os.path.exists(src_lbl) and not os.path.exists(dst_lbl):
            shutil.copytree(src_lbl, dst_lbl)

print("Dataset split done!")

Train: 87 books, Val: 22 books
Dataset split done!


In [ ]:


import os, shutil, yaml, json, time
import numpy as np
import torch
import torch.nn as nn
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.modules import Conv
import manga109api

data_root    = "/content/manga109_data/Manga109_released_2023_12_07"
dataset_root = "/content/manga109_dataset"
api          = manga109api.Parser(root_dir=data_root)

dataset_cfg = {
    "path":  dataset_root,
    "train": "images/train",
    "val":   "images/val",
    "nc":    2,
    "names": {0: "body", 1: "text"},
}
yaml_path = os.path.join(dataset_root, "manga109.yaml")
with open(yaml_path, "w") as f:
    yaml.dump(dataset_cfg, f)
print(f"Dataset YAML written → {yaml_path} ✓")


#blocks were taken from online
class SEBlock(nn.Module):

    def __init__(self, channels, reduction=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Sequential(
            nn.Linear(channels, max(channels // reduction, 4)),
            nn.SiLU(inplace=True),
            nn.Linear(max(channels // reduction, 4), channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        B, C, _, _ = x.shape
        w = self.pool(x).view(B, C)
        return x * self.fc(w).view(B, C, 1, 1)


class StyleRobustBottleneck(nn.Module):

    def __init__(self, in_ch, out_ch, shortcut=True):
        super().__init__()
        self.shortcut = shortcut and (in_ch == out_ch)
        self.cv1 = Conv(in_ch, out_ch, 1, 1)           # 1x1, keeps BN
        self.cv2 = nn.Sequential(                       # 3x3, IN instead of BN
            nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False),
            nn.InstanceNorm2d(out_ch, affine=True),
            nn.SiLU(inplace=True),
        )

    def forward(self, x):
        y = self.cv2(self.cv1(x))
        return x + y if self.shortcut else y


class StyleRobustC2f(nn.Module):

    def __init__(self, in_channels, out_channels, n=1, shortcut=True, se_reduction=16):
        super().__init__()
        mid = out_channels // 2
        self.cv1 = Conv(in_channels,   out_channels, 1, 1)
        self.cv2 = Conv((2 + n) * mid, out_channels, 1)
        self.bottlenecks = nn.ModuleList(
            StyleRobustBottleneck(mid, mid, shortcut) for _ in range(n)
        )
        self.se  = SEBlock(out_channels, reduction=se_reduction)
        self.mid = mid

    def forward(self, x):
        y = self.cv1(x)
        a, b = y.split(self.mid, dim=1)
        parts = [a, b]
        for m in self.bottlenecks:
            b = m(b)
            parts.append(b)
        return self.se(self.cv2(torch.cat(parts, dim=1)))


#AI WAS USED TO CREATE PATCHED TRAINER TO IMPLEMENT THESE MODULES
class PatchedTrainer(DetectionTrainer):
    def __init__(self, patched_model, overrides=None):
        self._patched_model = patched_model
        super().__init__(overrides=overrides)

    def get_model(self, cfg=None, weights=None, verbose=True):
        self._patched_model.nc = self.data['nc']
        return self._patched_model


# ── Load and inspect backbone C2f layers ──────────────────────────
# From earlier inspection, backbone C2f layers and their repeat counts:
#   Layer 2: C2f(64,  64,  n=1)  — read from [64, 64, 1, True]
#   Layer 4: C2f(128, 128, n=2)  — read from [128, 128, 2, True]
#   Layer 6: C2f(256, 256, n=2)  — read from [256, 256, 2, True]
#   Layer 8: C2f(512, 512, n=1)  — read from [512, 512, 1, True]

C2F_CONFIGS = {
    2: (64,  64,  1),
    4: (128, 128, 2),
    6: (256, 256, 2),
    8: (512, 512, 1),
}

base     = YOLO("yolov8s.pt")
detector = base.model

for layer_idx, (in_ch, out_ch, n) in C2F_CONFIGS.items():
    old = detector.model[layer_idx]
    new = StyleRobustC2f(in_ch, out_ch, n=n, shortcut=True)
    new.i = old.i
    new.f = old.f
    new.type = "StyleRobustC2f"
    detector.model[layer_idx] = new
    print(f"Layer {layer_idx} → StyleRobustC2f(in={in_ch}, out={out_ch}, n={n}) ✓")

# ── Verify forward pass ────────────────────────────────────────────
detector.cuda()
dummy = torch.randn(1, 3, 640, 640).cuda()
with torch.no_grad():
    detector(dummy)
print("Forward pass OK ✓")

# ── Layer table ───────────────────────────────────────────────────
print("\nLayer verification:")
for i, layer in enumerate(detector.model):
    marker = " ← CUSTOM" if type(layer).__name__ == 'StyleRobustC2f' else ""
    print(f"  Layer {i:2d} | {type(layer).__name__:22s}{marker}")

total = sum(p.numel() for p in detector.parameters())
print(f"\nTotal params: {total/1e6:.2f}M")

# ── Train ──────────────────────────────────────────────────────────
args = dict(
    data         = yaml_path,
    epochs       = 20,
    imgsz        = 640,
    batch        = 8,
    name         = "manga109_v2_style_c2f",
    project      = "/content/runs",
    device       = 0,
    optimizer    = "AdamW",
    lr0          = 5e-4,
    warmup_epochs= 3,
    hsv_s        = 0.5,
    mixup        = 0.1,
    model        = "yolov8s.pt",
)

trainer = PatchedTrainer(patched_model=detector, overrides=args)
trainer.train()
print("Training complete ✓")


Dataset YAML written → /content/manga109_dataset/manga109.yaml ✓
Layer 2 → StyleRobustC2f(in=64, out=64, n=1) ✓
Layer 4 → StyleRobustC2f(in=128, out=128, n=2) ✓
Layer 6 → StyleRobustC2f(in=256, out=256, n=2) ✓
Layer 8 → StyleRobustC2f(in=512, out=512, n=1) ✓
Forward pass OK ✓

Layer verification:
  Layer  0 | Conv                  
  Layer  1 | Conv                  
  Layer  2 | StyleRobustC2f         ← CUSTOM
  Layer  3 | Conv                  
  Layer  4 | StyleRobustC2f         ← CUSTOM
  Layer  5 | Conv                  
  Layer  6 | StyleRobustC2f         ← CUSTOM
  Layer  7 | Conv                  
  Layer  8 | StyleRobustC2f         ← CUSTOM
  Layer  9 | SPPF                  
  Layer 10 | Upsample              
  Layer 11 | Concat                
  Layer 12 | C2f                   
  Layer 13 | Upsample              
  Layer 14 | Concat                
  Layer 15 | C2f                   
  Layer 16 | Conv                  
  Layer 17 | Concat                
  Layer 18 | C2f  

In [ ]:
import os

# ── Evaluate ───────────────────────────────────────────────────────
weights_path = os.path.join(trainer.save_dir, "weights", "best.pt")
ckpt         = torch.load(weights_path, map_location='cpu', weights_only = False)
detector_eval = (ckpt.get('ema') or ckpt['model']).float().cuda()

eval_yolo       = YOLO("yolov8s.pt")
eval_yolo.model = detector_eval
metrics         = eval_yolo.val(data=yaml_path, imgsz=640, batch=8)

map50             = metrics.box.map50
map5095           = metrics.box.map
map50_per_class   = metrics.box.ap50
map5095_per_class = metrics.box.ap
precision         = metrics.box.mp
recall            = metrics.box.mr
f1                = 2 * (precision * recall) / (precision + recall + 1e-8)

print(f"\nmAP50:     {map50:.4f}")
print(f"mAP50-95:  {map5095:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1:        {f1:.4f}")

# ── Inference speed ───────────────────────────────────────────────
sample_img = os.path.join(data_root, "images", api.books[0], "002.jpg")
for _ in range(3):
    eval_yolo(sample_img, verbose=False)

times = []
for _ in range(50):
    start = time.perf_counter()
    eval_yolo(sample_img, verbose=False)
    times.append((time.perf_counter() - start) * 1000)

avg_ms = np.mean(times)
fps    = 1000 / avg_ms
print(f"Latency: {avg_ms:.2f} ms | FPS: {fps:.1f}")

# ── Save results ──────────────────────────────────────────────────
model_size_mb = os.path.getsize(weights_path) / (1024 ** 2)
param_count   = sum(p.numel() for p in detector_eval.parameters())

results = {
    "model":         "YOLOv8s_V2_StyleRobustC2f",
    "mAP50":         round(map50, 4),
    "mAP50-95":      round(map5095, 4),
    "precision":     round(precision, 4),
    "recall":        round(recall, 4),
    "f1":            round(f1, 4),
    "AP50_body":     round(map50_per_class[0], 4),
    "AP50_text":     round(map50_per_class[1], 4),
    "AP5095_body":   round(map5095_per_class[0], 4),
    "AP5095_text":   round(map5095_per_class[1], 4),
    "latency_ms":    round(avg_ms, 2),
    "fps":           round(fps, 1),
    "model_size_mb": round(model_size_mb, 2),
    "params_M":      round(param_count / 1e6, 2),
}

print("\n── Summary ──")
for k, v in results.items():
    print(f"  {k}: {v}")

with open("/content/results_v2_style_c2f.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved → /content/results_v2_style_c2f.json")

# ── Save to Drive ─────────────────────────────────────────────────
from google.colab import drive
import shutil

drive.mount('/drive')

shutil.copytree(
    str(trainer.save_dir),
    "/drive/MyDrive/manga109_v2_style_c2f",
    dirs_exist_ok=True
)

drive_results_dir = "/drive/MyDrive/CV-Comic-Project"
os.makedirs(drive_results_dir, exist_ok=True)
shutil.copy(
    "/content/results_v2_style_c2f.json",
    os.path.join(drive_results_dir, "results_v2_style_c2f.json")
)
print("Saved to Drive ✓")

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8s summary (fused): 105 layers, 10,341,596 parameters, 0 gradients, 26.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2844.8±810.7 MB/s, size: 382.5 KB)
val: Scanning /content/manga109_dataset/labels/val/TapkunNoTanteisitsu.cache... 2077 images, 94 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2077/2077 544.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 260/260 5.7it/s 45.6s
                   all       2077      59398      0.889       0.82      0.897      0.644
                  body       1979      31178      0.867      0.735      0.851       0.58
                  text       1962      28220      0.912      0.905      0.943      0.707
Speed: 1.5ms preprocess, 7.1ms inference, 0.0ms loss, 2.6ms postprocess per image
Results saved to /content/runs/detect/val-5

mAP50:     0.8966
mAP50-95:  0.6437
Precision: 0.8892
Recal

In [ ]:
# ── Evaluate ───────────────────────────────────────────────────────
weights_path = os.path.join(trainer.save_dir, "weights", "best.pt")
print(f"\nLoading best weights from {weights_path}")

# Reload patched model with best weights for eval
ckpt = torch.load(weights_path, map_location='cpu', weights_only=False)
detector_eval = ckpt.get('ema') or ckpt['model']
detector_eval = detector_eval.float().cuda()

# Wrap in YOLO for .val()
eval_yolo       = YOLO("yolov8s.pt")
eval_yolo.model = detector_eval
metrics         = eval_yolo.val(data=yaml_path, imgsz=640, batch=8)

map50             = metrics.box.map50
map5095           = metrics.box.map
map50_per_class   = metrics.box.ap50
map5095_per_class = metrics.box.ap
precision         = metrics.box.mp
recall            = metrics.box.mr
f1                = 2 * (precision * recall) / (precision + recall + 1e-8)

print(f"\nmAP50:     {map50:.4f}")
print(f"mAP50-95:  {map5095:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1:        {f1:.4f}")

# ── Inference speed ───────────────────────────────────────────────
sample_img = os.path.join(data_root, "images", api.books[0], "002.jpg")
for _ in range(3):
    eval_yolo(sample_img, verbose=False)

times = []
for _ in range(50):
    start = time.perf_counter()
    eval_yolo(sample_img, verbose=False)
    times.append((time.perf_counter() - start) * 1000)

avg_ms = np.mean(times)
fps    = 1000 / avg_ms
print(f"Latency: {avg_ms:.2f} ms | FPS: {fps:.1f}")

# ── Save results ──────────────────────────────────────────────────
model_size_mb = os.path.getsize(weights_path) / (1024 ** 2)
param_count   = sum(p.numel() for p in detector_eval.parameters())

results = {
    "model":         "YOLOv8s_V2_StyleRobustC2f",
    "mAP50":         round(map50, 4),
    "mAP50-95":      round(map5095, 4),
    "precision":     round(precision, 4),
    "recall":        round(recall, 4),
    "f1":            round(f1, 4),
    "AP50_body":     round(map50_per_class[0], 4),
    "AP50_text":     round(map50_per_class[1], 4),
    "AP5095_body":   round(map5095_per_class[0], 4),
    "AP5095_text":   round(map5095_per_class[1], 4),
    "latency_ms":    round(avg_ms, 2),
    "fps":           round(fps, 1),
    "model_size_mb": round(model_size_mb, 2),
    "params_M":      round(param_count / 1e6, 2),
}

print("\n── Summary ──")
for k, v in results.items():
    print(f"  {k}: {v}")

with open("/content/results_v2_style_c2f.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved → /content/results_v2_style_c2f.json")


# ── Save to Drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/drive')

shutil.copytree(
    str(trainer.save_dir),
    "/drive/MyDrive/manga109_v2_style_c2f",
    dirs_exist_ok=True
)
shutil.copy(
    "/content/results_v2_style_c2f.json",
    "/drive/MyDrive/CV-Comic-Project-results_v2_style_c2f.json"
)
print("Saved to Drive ✓")


Loading best weights from /content/runs/manga109_v2_style_c2f/weights/best.pt
Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8s summary (fused): 105 layers, 10,341,596 parameters, 0 gradients, 26.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2018.4±1073.2 MB/s, size: 357.4 KB)
val: Scanning /content/manga109_dataset/labels/val/TapkunNoTanteisitsu.cache... 2077 images, 94 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2077/2077 622.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 260/260 6.2it/s 41.8s
                   all       2077      59398      0.889       0.82      0.897      0.644
                  body       1979      31178      0.867      0.735      0.851       0.58
                  text       1962      28220      0.912      0.905      0.943      0.707
Speed: 1.3ms preprocess, 6.5ms inference, 0.0ms loss, 2.3ms postprocess per image
Results saved to /conten